### Load packages

In [3203]:
import pandas as pd
# import numpy as np
import os
# import networkx as nx
import plotly.graph_objects as go
import plotly.io as pio

### Define parameters and file info

In [3204]:
pd.set_option('display.max_colwidth', None)

In [3205]:
data_directory = r"..\data"

# -----------------------------------------------------------------------
# CSV data
# -----------------------------------------------------------------------

node_file = r"MCN_nonprofit_economy_revenue_nodes_2023.csv"
node_path = os.path.join(data_directory, node_file)
print("Node CSV file:", node_path)
if os.path.exists(node_path):
    print("EXISTS")

edge_file = r"MCN_nonprofit_economy_revenue_edges_2023.csv"
edge_path = os.path.join(data_directory, edge_file)
print("Edge CSV file:", edge_path)
if os.path.exists(edge_path):
    print("EXISTS")

# -----------------------------------------------------------------------
# Output
# -----------------------------------------------------------------------

fig_output_file = "MCN_nonprofit_economy_sankey_2023"
output_directory = r"..\output"

Node CSV file: ..\data\MCN_nonprofit_economy_revenue_nodes_2023.csv
EXISTS
Edge CSV file: ..\data\MCN_nonprofit_economy_revenue_edges_2023.csv
EXISTS


### Load data

In [3206]:
# Load nonprofit economy node data

node_raw_df = pd.read_csv(node_path)
node_raw_df.head()

,Node,Node Type,X,Y,X Offset,Y Offset
0,Donor-advised fund sponsors (national and community foundation only),Intermediary Right,0.60,0.05,0.03,-0.007
1,Donor-advised fund sponsors (national and community foundation only) (ghost),Ghost,0.65,0.05,0.00,0.000
2,Foundations (minus community foundation DAF sponsors),Intermediary Right,0.33,0.07,0.02,-0.009
3,Foundations (minus community foundation DAF sponsors) (ghost),Ghost,0.38,0.07,0.00,0.000
4,Program fees from private sources,Source,0.10,0.86,0.02,-0.037


In [3207]:
# Load nonprofit economy edge data

edge_raw_df = pd.read_csv(edge_path)
edge_raw_df.head()

,Source,Recipient,Amount
0,Donor-advised fund sponsors (national and community foundation only),"Arts, culture, humanities",3.42
1,Donor-advised fund sponsors (national and community foundation only),Education (minus colleges and universities),6.99
2,Donor-advised fund sponsors (national and community foundation only),Colleges and universities,6.82
3,Donor-advised fund sponsors (national and community foundation only),Environment and animals,3.41
4,Donor-advised fund sponsors (national and community foundation only),Health (minus hospitals and nursing homes),3.49


In [3208]:
# Clean & format columns in edge data

edge_clean_df = edge_raw_df.copy()

edge_clean_df["Amount"] = pd.to_numeric(
    edge_clean_df["Amount"].replace("-", 0),
    errors="coerce"
)

edge_clean_df.head()

,Source,Recipient,Amount
0,Donor-advised fund sponsors (national and community foundation only),"Arts, culture, humanities",3.42
1,Donor-advised fund sponsors (national and community foundation only),Education (minus colleges and universities),6.99
2,Donor-advised fund sponsors (national and community foundation only),Colleges and universities,6.82
3,Donor-advised fund sponsors (national and community foundation only),Environment and animals,3.41
4,Donor-advised fund sponsors (national and community foundation only),Health (minus hospitals and nursing homes),3.49


### Clean up data in dataframes

In [3209]:
node_raw_df.head()

,Node,Node Type,X,Y,X Offset,Y Offset
0,Donor-advised fund sponsors (national and community foundation only),Intermediary Right,0.60,0.05,0.03,-0.007
1,Donor-advised fund sponsors (national and community foundation only) (ghost),Ghost,0.65,0.05,0.00,0.000
2,Foundations (minus community foundation DAF sponsors),Intermediary Right,0.33,0.07,0.02,-0.009
3,Foundations (minus community foundation DAF sponsors) (ghost),Ghost,0.38,0.07,0.00,0.000
4,Program fees from private sources,Source,0.10,0.86,0.02,-0.037


In [3210]:
# Function for setting node totals

def get_total(row):
    node = row["Node"]
    node_type = row["Node Type"]

    if node_type == "Source":
        amount = edge_clean_df.loc[
            edge_clean_df["Source"] == node,
            "Amount"
        ].sum()

        return f"${amount:,.0f}B"

    elif node_type == "Recipient":
        amount = edge_clean_df.loc[
            edge_clean_df["Recipient"] == node,
            "Amount"
        ].sum()

        return f"${amount:,.0f}B"

    elif node_type == "Intermediary Right":
        incoming = edge_clean_df.loc[
            edge_clean_df["Recipient"] == node,
            "Amount"
        ].sum()

        outgoing = edge_clean_df.loc[
            edge_clean_df["Source"] == node,
            "Amount"
        ].sum()

        return f"In: ${incoming:,.0f}B<br>Out: ${outgoing:,.0f}B"

    return ""

In [3211]:
# Add column for total in or out of node

node_totals_df = node_raw_df.copy()

node_totals_df["Total"] = node_totals_df.apply(get_total, axis=1)

node_totals_df.head(10)

,Node,Node Type,X,Y,X Offset,Y Offset,Total
0,Donor-advised fund sponsors (national and community foundation only),Intermediary Right,0.60,0.05,0.03,-0.0070,In: $76B<br>Out: $54B
1,Donor-advised fund sponsors (national and community foundation only) (ghost),Ghost,0.65,0.05,0.00,0.0000,
2,Foundations (minus community foundation DAF sponsors),Intermediary Right,0.33,0.07,0.02,-0.0090,In: $178B<br>Out: $104B
3,Foundations (minus community foundation DAF sponsors) (ghost),Ghost,0.38,0.07,0.00,0.0000,
4,Program fees from private sources,Source,0.10,0.86,0.02,-0.0370,"$1,923B"
5,Federal government,Source,0.10,0.77,0.02,-0.0330,$469B
6,State and local government,Source,0.10,0.68,0.02,-0.0350,$204B
7,Federated Giving,Source,0.10,0.59,0.02,-0.0012,$3B
8,Investment Income,Source,0.10,0.50,0.02,-0.0010,$67B
9,Corporations,Source,0.10,0.41,0.02,0.0010,$37B


In [3212]:
# Function to find the middle of a string and add a <br> at the space that is closest to that wrap_at_middle

def wrap_at_middle(text):
    if pd.isna(text):
        return text
    
    # Find positions of all spaces
    space_positions = [i for i, char in enumerate(text) if char == " "]
    
    # If no spaces, return unchanged
    if not space_positions:
        return text
    
    # Midpoint of string
    midpoint = len(text) / 2
    
    # Find space closest to midpoint
    split_pos = min(space_positions, key=lambda x: abs(x - midpoint))
    
    # Insert <br>
    return text[:split_pos] + "<br>" + text[split_pos + 1:]

In [3213]:
# add extra columns for node name formatting
node_colname_df = node_totals_df.copy()

node_colname_df["Node Short"] = node_colname_df["Node"].replace({
    "Donor-advised fund sponsors (national and community foundation only)": "National & CF DAF sponsors",
    "Donor-advised fund sponsors (national and community foundation only) (ghost)": "National & CF DAF sponsors (ghost)",
    "Foundations (minus community foundation DAF sponsors)": "Foundations",
    "Foundations (minus community foundation DAF sponsors) (ghost)": "Foundations (ghost)",
    "Program fees from private sources": "Program fees",
    "State and local government": "State & local government",
    "Federated Giving": "Federated giving",
    "Hospitals and nursing homes": "Hospitals & nursing homes",
    "Health (minus hospitals and nursing homes)": "Other healthcare",
    "Education (minus colleges and universities)": "Other education",
    "Public/societal benefit (minus national DAF sponsors)": "Public/societal benefit",
    "Arts, culture, humanities": "Arts & culture",
    "Environment and animals": "Environment & animals",
    "International/foreign affairs": "International/ foreign affairs",
    "Unknown, unclassified": "Unknown"
})

In [3224]:
# Create a wrapped node text column
node_format_df = node_colname_df.copy()

node_format_df["Node Wrapped"] = node_format_df["Node Short"].apply(wrap_at_middle)

conditions = [
    node_format_df["Node Type"] == "Source",
    node_format_df["Node Type"] == "Recipient",
    node_format_df["Node Type"].isin(["Intermediary Right", "Intermediary Left"])
]

choices = [
    "From<br>" + node_format_df["Node Wrapped"] + "<br>" + node_format_df["Total"].astype(str),
    "To<br>" + node_format_df["Node Wrapped"] + "<br>" + node_format_df["Total"].astype(str),
    node_format_df["Node Wrapped"] + "<br>" + node_format_df["Total"].astype(str)
]

node_format_df["Node Totals"] = np.select(
    conditions,
    choices,
    default=node_format_df["Node Wrapped"]
)

node_format_df.head(20)

,Node,Node Type,X,Y,X Offset,Y Offset,Total,Node Short,Node Wrapped,Node Totals
0,Donor-advised fund sponsors (national and community foundation only),Intermediary Right,0.60,0.05,0.03,-0.0070,In: $76B<br>Out: $54B,National & CF DAF sponsors,National & CF<br>DAF sponsors,National & CF<br>DAF sponsors<br>In: $76B<br>Out: $54B
1,Donor-advised fund sponsors (national and community foundation only) (ghost),Ghost,0.65,0.05,0.00,0.0000,,National & CF DAF sponsors (ghost),National & CF DAF<br>sponsors (ghost),National & CF DAF<br>sponsors (ghost)
2,Foundations (minus community foundation DAF sponsors),Intermediary Right,0.33,0.07,0.02,-0.0090,In: $178B<br>Out: $104B,Foundations,Foundations,Foundations<br>In: $178B<br>Out: $104B
3,Foundations (minus community foundation DAF sponsors) (ghost),Ghost,0.38,0.07,0.00,0.0000,,Foundations (ghost),Foundations<br>(ghost),Foundations<br>(ghost)
4,Program fees from private sources,Source,0.10,0.86,0.02,-0.0370,"$1,923B",Program fees,Program<br>fees,"From<br>Program<br>fees<br>$1,923B"
5,Federal government,Source,0.10,0.77,0.02,-0.0330,$469B,Federal government,Federal<br>government,From<br>Federal<br>government<br>$469B
6,State and local government,Source,0.10,0.68,0.02,-0.0350,$204B,State & local government,State & local<br>government,From<br>State & local<br>government<br>$204B
7,Federated Giving,Source,0.10,0.59,0.02,-0.0012,$3B,Federated giving,Federated<br>giving,From<br>Federated<br>giving<br>$3B
8,Investment Income,Source,0.10,0.50,0.02,-0.0010,$67B,Investment Income,Investment<br>Income,From<br>Investment<br>Income<br>$67B
9,Corporations,Source,0.10,0.41,0.02,0.0010,$37B,Corporations,Corporations,From<br>Corporations<br>$37B


### Prepare data for plotting

In [3215]:
# Select node field to use

node_display_field = "Node Totals"
node_id_field = "Node"

In [3216]:
# Prepare data for plotting

# Build a node map (convert node labels into integer indices)
node_map = {
    node: i for i, node in enumerate(node_format_df[node_id_field])
}

# Convert source and recipient node into indices
sources = edge_clean_df["Source"].map(node_map)
targets = edge_clean_df["Recipient"].map(node_map)
values = edge_clean_df["Amount"]

In [3217]:
edge_clean_df["Source"].unique()

array(['Donor-advised fund sponsors (national and community foundation only)',
       'Donor-advised fund sponsors (national and community foundation only) (ghost)',
       'Individuals',
       'Foundations (minus community foundation DAF sponsors)',
       'Foundations (minus community foundation DAF sponsors) (ghost)',
       'Bequests', 'Federated Giving', 'Corporations',
       'State and local government', 'Federal government',
       'Program fees from private sources', 'Investment Income'],
      dtype=object)

In [3218]:
# Map colors to edges and nodes

source_colors = {
    "Program fees from private sources": "rgba(160,160,160,0.35)",
    "Federal government": "rgba(230,110,110,0.35)",
    "State and local government": "rgba(230,110,110,0.35)",
    "Individuals": "rgba(31,119,180,0.35)",
    "Bequests": "rgba(255,127,14,0.35)",
    "Federated Giving": "rgba(44,160,44,0.35)",
    "Corporations": "rgba(214,39,40,0.35)",
    "Investment Income": "rgba(44,160,44,0.35)",
    "Donor-advised fund sponsors (national and community foundation only)": "rgba(200,0,0,0.35)",
    "Donor-advised fund sponsors (national and community foundation only) (ghost)": "rgba(200,0,0,0.35)",
    "Foundations (minus community foundation DAF sponsors)": "rgba(0,0,200,0.35)",
    "Foundations (minus community foundation DAF sponsors) (ghost)": "rgba(0,0,200,0.35)"
}
edge_colors = (
    edge_clean_df["Source"]
    .map(source_colors)
    .fillna("rgba(100,100,100,0.35)")
)

node_colors = []

node_colors = node_format_df.apply(
    lambda row: "rgba(0,0,0,0)"
    if row["Node Type"] == "Ghost"
    else source_colors.get(row["Node"], "rgba(100,100,100,0.35)"),
    axis=1
)

In [3219]:
# Create custom node names 
node_names = node_format_df[node_display_field].tolist()

node_names

['National & CF<br>DAF sponsors<br>In: $76B<br>Out: $54B',
 'National & CF DAF<br>sponsors (ghost)',
 'Foundations<br>In: $178B<br>Out: $104B',
 'Foundations<br>(ghost)',
 'From<br>Program<br>fees<br>$1,923B',
 'From<br>Federal<br>government<br>$469B',
 'From<br>State & local<br>government<br>$204B',
 'From<br>Federated<br>giving<br>$3B',
 'From<br>Investment<br>Income<br>$67B',
 'From<br>Corporations<br>$37B',
 'From<br>Bequests<br>$43B',
 'From<br>Individuals<br>$366B',
 'To<br>Hospitals &<br>nursing homes<br>$1,299B',
 'To<br>Other<br>healthcare<br>$329B',
 'To<br>Human<br>Services<br>$346B',
 'To<br>Colleges and<br>universities<br>$307B',
 'To<br>Other<br>education<br>$207B',
 'To<br>Public/societal<br>benefit<br>$215B',
 'To<br>Religious<br>congregations<br>$152B',
 'To<br>International/<br>foreign affairs<br>$56B',
 'To<br>Arts &<br>culture<br>$57B',
 'To<br>Environment<br>& animals<br>$35B',
 'To<br>Unknown<br>$13B']

In [3220]:
# Build hover display fields
hover_text = []

for s, t, v in zip(sources, targets, values):
    source_name = node_names[s]
    target_name = node_names[t]

    hover_text.append(
        f"From: {source_name}<br>"
        f"To: {target_name}<br>"
        f"Amount: ${v:,.0f}"
    )

### Visualize with Plotly Sankey (static charts)

Using Plotly Sankey because it includes the best compromises for:
- weighted edges
- self-loops (still poor)
- curved edges
- label control (still poor)
- heirarchy levels
- interactivity
- quality
- merging flows

In [3221]:
# Plot the diagram
fig = go.Figure(go.Sankey(
    arrangement="snap",
    
    node=dict(
        # label=node_format_df["Node"],
        label=[""] * len(node_format_df), # remove default labels
        x=node_format_df["X"],
        y=node_format_df["Y"],
        pad=15,
        thickness=18,
        line=dict(color="black", width=0.5),
        # line=dict(color="rgba(0,0,0,0)", width=0),
        color=node_colors
    ),
    
    link=dict(
        # arrowlen=10,
        source=sources,
        target=targets,
        value=values,
        color=edge_colors,
        customdata=hover_text,
        hovertemplate="%{customdata}<extra></extra>"
    )
))

# Define annotations for labels
annotations = []

for _, row in node_format_df[node_format_df["Node Type"] == "Source"].iterrows():
    node_name = row[node_id_field]
    dx = row["X Offset"]
    dy = row["Y Offset"]
    annotations.append(dict(
        x=row["X"] - dx,
        y=(1 - row["Y"]) + dy,
        text=row[node_display_field],
        showarrow=False,
        xanchor="right",
        align="center",
        textangle=-90
    ))

for _, row in node_format_df[node_format_df["Node Type"] == "Recipient"].iterrows():
    dx = row["X Offset"]
    dy = row["Y Offset"]
    annotations.append(dict(
        x=row["X"] + dx,
        y=(1 - row["Y"]) + dy,
        text=row[node_display_field],
        showarrow=False,
        xanchor="left",
        align="center",
        textangle=-90
    ))
for _, row in node_format_df[node_format_df["Node Type"] == "Intermediary Left"].iterrows():
    dx = row["X Offset"]
    dy = row["Y Offset"]
    annotations.append(dict(
        x=row["X"] - dx,
        y=(1 - row["Y"]) + dy,
        text=row[node_display_field],
        showarrow=False,
        xanchor="left",
        align="center",
        textangle=-90
    ))

for _, row in node_format_df[node_format_df["Node Type"] == "Intermediary Right"].iterrows():
    dx = row["X Offset"]
    dy = row["Y Offset"]
    annotations.append(dict(
        x=row["X"] + dx,
        y=(1 - row["Y"]) + dy,
        text=row[node_display_field],
        showarrow=False,
        xanchor="right",
        align="center",
        textangle=-90
    ))

# Format
fig.update_layout(
    # title_text="Nonprofit Economy Flows (2023)",
    font_size=10,
    width=800,
    height=1100,
    margin=dict(l=20, r=20, t=40, b=20),
    annotations=annotations
)

fig.show()

### Output image

In [3222]:

# Output as PNG
fig_png_output_path = os.path.join(output_directory, fig_output_file + ".png")
# fig.write_image(fig_output_path)

# Output as SVG
fig_svg_output_path = os.path.join(output_directory, fig_output_file + ".svg")
# fig.write_image(fig_svg_output_path)

In [3223]:
# Output to HTML

fig_html_output_path = os.path.join(output_directory, fig_output_file + ".html")

html_fragment = pio.to_html(fig, full_html=False, include_plotlyjs="cdn")

full_html = f"""
<html>
<head>
<meta charset="utf-8" />
<style>
.rotated-wrapper {{
    position: relative;
    width: 1100px;
    height: 800px;
}}

.rotated-plot {{
    position: absolute;
    top: 0;
    left: 0;
    transform: rotate(90deg) translateY(-100%) translateX(0px);
    transform-origin: top left;
}}
</style>
</head>
<body>
<div class="rotated-wrapper">
<div class="rotated-plot">
{html_fragment}
</div>
</div>
</body>
</html>
"""

with open(fig_html_output_path, "w", encoding="utf-8") as f:
    f.write(full_html)